# The frozen 20 K rule

The score is the hottest usable band-7 pixel within 20 km minus the median of that maximum on up to five previous covering scans. At least three previous scans are required. `THRESHOLD_K` is the constant 20.0, chosen from the Flight 11 measurements. Nothing in this notebook refits it, and nothing here downloads a file.

A candidate is emitted, then held back on the third consecutive covering scan at or above 20 K. ABI-L2-FDCM codes 30–35 within 20 km of the hot pixel also hold it. Codes 10–15 are logged and do not hold: the Flight 11 plume itself received code 10. The walkthrough is `docs/scoring.md`.


In [ ]:
import json
import sys
from pathlib import Path

here = Path.cwd()
root = here if (here / "pyproject.toml").exists() else here.parent
sys.path.insert(0, str(root))

from monitorear.goes_io import ROOT
from monitorear.score import THRESHOLD_K, self_test_rule

print(f"THRESHOLD_K = {THRESHOLD_K}")
self_test_rule()

jsonl = ROOT / "outputs" / "m3_candidates.jsonl"
if not jsonl.exists():
    print("No candidate log yet. Run scripts/m3_score.py")
else:
    for line in jsonl.read_text().splitlines():
        row = json.loads(line)
        print(
            row.get("scan_start"),
            row.get("watch"),
            row.get("status"),
            row.get("score_k"),
        )


The self-test locks the arithmetic that makes the second hot scan still clear the cut: a median of five previous maxima stays on the pre-liftoff samples when only the newest of them is hot. Flight 11 therefore emits two scans, 39.29 K at 23:23:54.9 UTC and 45.37 K at 23:24:54.9 UTC. The quiet evening peaks at 0.04 K under this trailing median. The Georgia fire, scored with the same rule and not armed as a watch site, peaks at 12.36 K. Re-running the launch only checks that the rule still emits the scans it was drawn around.
